In [ ]:
# # 03 - Profit Optimisation
# Contribution per order = (food price - $4) + (delivery fee - $0.50 x miles).
# Demand is constant-elasticity, calibrated on each treated region's pre-policy mean.
# Fixed costs ($5,000/yr, about $13.70/day) do not change the optimum, so profits here
# are contribution margins.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import ROOT, load_orders, did_sample
from src import models, optimise

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})
FIG = ROOT / "figures"
CAPACITY = 22                    # ASSUMPTION: ~max daily orders seen in any region
MAX_DISCOUNT = 0.15              # trust region: no more than 3x the tested price change
DIST = {3: 9, 4: 15}
LEVERS = [(3, "food"), (4, "fee")]

df = load_orders()
res = {}
for region, lever in LEVERS:
    d = models.did_log(did_sample(df, region))
    base_q = df[(df.region == region) & (df.post == 0)].qty.mean()
    res[lever] = dict(did=d, base_q=base_q)

In [ ]:
# ## Step 1: the textbook answer (Inverse Elasticity Rule)
# `P* = MC / (1 + 1/eta)`. For the fee lever, the effective MC nets off the pizza margin
# earned on the extra orders a cheaper delivery brings in, so it can be negative.

# %%
for region, lever in LEVERS:
    eta = res[lever]["did"]["elasticity"]
    mc = optimise.effective_mc(lever, DIST[region])
    print(f"Region {region} ({lever}): effective MC = {mc:6.2f}, eta = {eta:6.2f}, "
          f"IER price = {optimise.ier_price(mc, eta):6.2f}")

# With |eta| above 12, the rule pushes the food price close to marginal cost and the fee
# below zero. Both imply an unbounded number of orders, which no kitchen can serve, and
# both are far outside the 5% change the elasticity was measured on. That is an
# extrapolation artifact, so we constrain the search.


In [ ]:
# ## Step 2: constrained optimum
# Search discounts from -15% to +15% (negative = price increase) with a capacity ceiling.

rows = []
for region, lever in LEVERS:
    base_q, d = res[lever]["base_q"], res[lever]["did"]
    lo, hi = d["elasticity_ci"]
    base = optimise.profit_curve(lever, base_q, d["elasticity"], DIST[region], [0.0]).profit[0]
    cur = optimise.profit_curve(lever, base_q, d["elasticity"], DIST[region], [0.05], CAPACITY).profit[0]
    for label, eta in [("point", d["elasticity"]), ("CI low", lo), ("CI high", hi)]:
        b = optimise.best_in_trust_region(lever, base_q, eta, DIST[region], MAX_DISCOUNT, CAPACITY)
        rows.append({"region": region, "lever": lever, "elasticity case": label,
                     "eta": round(eta, 1), "best discount": f"{b.discount:.1%}",
                     "orders/day": round(b.orders, 1), "profit/day": round(b.profit, 2),
                     "5% profit": round(cur, 2), "no-discount profit": round(base, 2)})
pd.DataFrame(rows).set_index(["region", "elasticity case"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grid = np.linspace(-0.05, MAX_DISCOUNT, 81)
for ax, (r, lever) in zip(axes, LEVERS):
    for cap, ls in [(None, "-"), (CAPACITY, "--")]:
        c = optimise.profit_curve(lever, res[lever]["base_q"], res[lever]["did"]["elasticity"],
                                  DIST[r], grid, cap)
        ax.plot(c.discount * 100, c.profit, ls, label=f"capacity = {cap}")
    ax.axvline(5, ls=":", color="k"); ax.set_yscale("log")
    ax.set_title(f"Region {r}: profit vs {lever} discount")
    ax.set_xlabel("Discount (%)"); ax.set_ylabel("Daily contribution ($, log scale)")
    ax.legend()
plt.tight_layout(); plt.savefig(FIG / "profit_curves.png", dpi=150); plt.show()

In [ ]:
# ## Step 3: how much does the capacity assumption matter?

caps = [16, 18, 20, 22, 25, 30]
sens = []
for region, lever in LEVERS:
    for cap in caps:
        b = optimise.best_in_trust_region(lever, res[lever]["base_q"],
                                          res[lever]["did"]["elasticity"], DIST[region],
                                          MAX_DISCOUNT, cap)
        sens.append({"lever": lever, "capacity": cap, "best discount": b.discount * 100})
sens = pd.DataFrame(sens).pivot(index="capacity", columns="lever", values="best discount")
ax = sens.plot(marker="o"); ax.set_ylabel("Best discount (%)")
ax.set_title("Optimal discount vs assumed capacity")
plt.tight_layout(); plt.savefig(FIG / "capacity_sensitivity.png", dpi=150); plt.show()
sens.round(1)

In [ ]:
# ## Conclusions
# - Both discounts lifted orders, and the DiD diagnostics pass.
# - Constant-elasticity demand with |eta| > 12 has no sensible unconstrained optimum, so
#   a point recommendation cannot be defended from this data alone.
# - With a capacity of about 22 orders/day, the best discounts are roughly 5-9% (food)
#   and 6-13% (fee). They are close to what was tested and move with the capacity
#   assumption (see the sensitivity chart).
# - Recommendation: run a randomised multi-level test (0%, 5%, 10%, 15%) and re-estimate
#   before scaling any discount.